Versión mejorada con ayuda de https://www.kaggle.com/code/wissams/titanic-competition-step-by-step-using-xgboost/notebook.

In this competition, you’ll gain access to two similar datasets that include passenger information like name, age, gender, socio-economic class, etc. One dataset is titled "train.csv" and the other is titled "test.csv".

"train.csv" will contain the details of a subset of the passengers on board (891 to be exact) and importantly, will reveal whether they survived or not, also known as the “ground truth”.

The "test.csv" dataset contains similar information but does not disclose the “ground truth” for each passenger. It’s your job to predict these outcomes.

Using the patterns you find in the "train.csv" data, predict whether the other 418 passengers on board (found in test.csv) survived.

In [132]:
import pandas as pd
import numpy as np

# dataset with the ground truth
data_train = pd.read_csv("/Users/alvaromartinezmartinez/Desktop/Python/MLJupyter/train_titanic.csv")

# dataset for which we need to predict the survival
data_test = pd.read_csv("/Users/alvaromartinezmartinez/Desktop/Python/MLJupyter/test_titanic.csv")

El objetivo es determinar "Survived" para cada "PassengerId" del fichero "train.csv".

## Gestión inicial de los datos

In [133]:
data_train.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

In [134]:
data_test.columns

Index(['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch',
       'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

In [135]:
data_train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [136]:
data_test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


Definición de variables:
- passengerid
- survival: 0 = No, 1 = Yes
- pclass (Ticket class): 1 = 1st, 2 = 2nd, 3 = 3rd
- name
- sex
- age: in years
- sibsp: # of siblings / spouses aboard the Titanic
- parch: # of parents / children aboard the Titanic
- ticket: ticket number
- fare
- cabin: cabin number
- embarked (port of embarkation): C = Cherbourg, Q = Queenstown, S = Southampton

## Tratamiento de los datos

Ahora quiero ver qué columnas tienen huecos vacíos para poder quitarlas o imputarlas.

In [137]:
# Shape of training data (num_rows, num_columns)
print(data_train.shape)

# Number of missing values in each column of training data
missing_val_count_by_column_train = (data_train.isnull().sum())
print(missing_val_count_by_column_train[missing_val_count_by_column_train > 0])

(891, 12)
Age         177
Cabin       687
Embarked      2
dtype: int64


In [138]:
# Shape of test data (num_rows, num_columns)
print(data_test.shape)

# Number of missing values in each column of test data
missing_val_count_by_column_test = (data_test.isnull().sum())
print(missing_val_count_by_column_test[missing_val_count_by_column_test > 0])

(418, 11)
Age       86
Fare       1
Cabin    327
dtype: int64


Vemos que la columna de cabina está practicamente vacía, pero seguramente sea porque los que no estaban en primera clase no tenían cabina asignada.

La de embarked está practicamente llena, pero son strings, así que quiero transformarla a los valores 0,1,2, e imputar los dos huecos vacíos con el puerto más frecuente.

Quito también las columnas de Ticket y Name ya que creo que no tienen tanta importancia (Ticket está relacionado con Fare).

La de edad la podemos imputar por clase y sexo.

In [139]:
# Drop ticket and name columns
reduced_data_train = data_train.drop(['Ticket'], axis=1) # 'Name'
reduced_data_test = data_test.drop(['Ticket'],axis=1)

Imputación del único valor de "Fare" que está libre. Imputo en función de 'Class' del hueco.

In [140]:
median_fare_per_class = reduced_data_test.groupby("Pclass")["Fare"].median()

def fill_fare_test(row):
    if pd.isnull(row["Fare"]):  # si la edad está vacía
        return median_fare_per_class.loc[row["Pclass"]]
    else:
        return row["Fare"]
    
# imputo el hueco de Fare del dataset test
reduced_data_test["Fare"] = reduced_data_test.apply(fill_fare_test, axis=1)

Imputacion de las edades por sexo y clase.

In [141]:
# edades medias agrupadas por clase y sexo
median_ages_train = reduced_data_train.groupby(["Pclass", "Sex"])["Age"].median()
median_ages_test = reduced_data_test.groupby(["Pclass", "Sex"])["Age"].median()

# funcion que sirve para imputar los huecos vacíos con las edades agrupadas por clase y sexo
def fill_age_train(row):
    if pd.isnull(row["Age"]):  # si la edad está vacía
        return median_ages_train.loc[row["Pclass"], row["Sex"]]
    else:
        return row["Age"]
    
def fill_age_test(row):
    if pd.isnull(row["Age"]):  # si la edad está vacía
        return median_ages_test.loc[row["Pclass"], row["Sex"]]
    else:
        return row["Age"]
    
# imputo los huecos
reduced_data_train["Age"] = reduced_data_train.apply(fill_age_train, axis=1)
reduced_data_test["Age"] = reduced_data_test.apply(fill_age_test, axis=1)

Ahora quiero hacer ordinal encoding tranformando Female = 0 y Male = 1.

In [142]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
reduced_data_train["Sex"] = encoder.fit_transform(reduced_data_train["Sex"])
reduced_data_test["Sex"] = encoder.fit_transform(reduced_data_test["Sex"])

Ahora voy a obtener la primera letra del "Cabin".

In [143]:
# obtengo la primera letra de Cabin y luego sustituyo la columna Cabin por Cabin_Letter
reduced_data_train['Cabin_Letter'] = reduced_data_train['Cabin'].str[:1]
reduced_data_train.Cabin_Letter.fillna('Unknown', inplace=True)
reduced_data_train= reduced_data_train.drop(columns=['Cabin'])

reduced_data_train.Cabin_Letter.value_counts()


/var/folders/qs/p4hy04495h79gqp0b41ths340000gn/T/ipykernel_8235/1649161886.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  reduced_data_train.Cabin_Letter.fillna('Unknown', inplace=True)


Cabin_Letter
Unknown    687
C           59
B           47
D           33
E           32
A           15
F           13
G            4
T            1
Name: count, dtype: int64

In [144]:
# Obtengo la primera letra de Cabin y luego sustituyo la columna Cabin por Cabin_Letter

reduced_data_test['Cabin_Letter'] = reduced_data_test['Cabin'].str[:1]
reduced_data_test.Cabin_Letter.fillna('Unknown', inplace=True)
reduced_data_test= reduced_data_test.drop(columns=['Cabin'])

reduced_data_test.Cabin_Letter.value_counts()

/var/folders/qs/p4hy04495h79gqp0b41ths340000gn/T/ipykernel_8235/2062025157.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  reduced_data_test.Cabin_Letter.fillna('Unknown', inplace=True)


Cabin_Letter
Unknown    327
C           35
B           18
D           13
E            9
F            8
A            7
G            1
Name: count, dtype: int64

Voy a leer los títulos de los nombres y a agrupar algunos y luego les hago ordinal encoding.

In [145]:
reduced_data_train['Title'] = reduced_data_train['Name'].str.split(', ').str[1].str.split('.').str[0]
reduced_data_test['Title'] = reduced_data_test['Name'].str.split(', ').str[1].str.split('.').str[0]

reduced_data_train['Title'].unique() # Muestro los títulos del set de train

array(['Mr', 'Mrs', 'Miss', 'Master', 'Don', 'Rev', 'Dr', 'Mme', 'Ms',
       'Major', 'Lady', 'Sir', 'Mlle', 'Col', 'Capt', 'the Countess',
       'Jonkheer'], dtype=object)

In [146]:
# Agrupacion de algunos titulos minoritarios

title_mapping= {'Mr':'Mr', 'Mrs':'Mrs', 'Miss':'Miss','Master':'Master', 'Don':'Rare', 'Rev':'Rare', 'Dr':'Rare', 'Mme':'Mrs', 'Ms':'Miss',
       'Major':'Rare', 'Lady':'Rare' , 'Sir':'Rare', 'Mlle':'Miss', 'Col':'Rare', 'Capt':'Rare', 'the Countess':'Rare',
       'Jonkheer':'Rare', 'Dona':'Rare'}

reduced_data_train['Title']= reduced_data_train['Title'].map(title_mapping)
reduced_data_test['Title']= reduced_data_test['Title'].map(title_mapping)

In [147]:
reduced_data_train['Title'].value_counts()

Title
Mr        517
Miss      185
Mrs       126
Master     40
Rare       23
Name: count, dtype: int64

In [148]:
reduced_data_test['Title'].value_counts()

Title
Mr        240
Miss       79
Mrs        72
Master     21
Rare        6
Name: count, dtype: int64

Ahora hago ordinal encoding de estos titulos:

Por último, lo que voy a hacer es juntar las columnas SibSp y Parch para crear la columna 'FamilySize' para generalizar más.

In [149]:
# +1 incluyendo al propio pasajero
reduced_data_train["FamilySize"] = reduced_data_train["Parch"] + reduced_data_train["SibSp"] + 1
reduced_data_test["FamilySize"] = reduced_data_test["Parch"] + reduced_data_test["SibSp"] + 1

Vamos a hacer One Hot Encoding a las columnas categóricas 'Cabin_Letter' y 'Title' para ver si mejora así la solución. Es importante hacer OHE ya que si se hace solo Ordinal Encoding, el modelo de ML puede crear relaciones falsas al haber un orden de números fictíceo.

In [150]:
from sklearn.preprocessing import OneHotEncoder

# Crear el encoder
OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit y transform al train
OH_reduced_train = pd.DataFrame(
    OH_encoder.fit_transform(reduced_data_train[['Cabin_Letter','Title','Embarked']]),
    index=reduced_data_train.index,
    columns=OH_encoder.get_feature_names_out(['Cabin_Letter','Title','Embarked'])
)

# Transform al test (con los mismos nombres de columnas)
OH_reduced_test = pd.DataFrame(
    OH_encoder.transform(reduced_data_test[['Cabin_Letter','Title','Embarked']]),
    index=reduced_data_test.index,
    columns=OH_encoder.get_feature_names_out(['Cabin_Letter','Title','Embarked']))

# Eliminar columnas originales categóricas
reduced_data_train = reduced_data_train.drop(['Cabin_Letter','Title','Embarked'], axis=1)
reduced_data_test = reduced_data_test.drop(['Cabin_Letter','Title','Embarked'], axis=1)

# Concatenar de nuevo las columnas OHE
reduced_data_train = pd.concat([reduced_data_train, OH_reduced_train], axis=1)
reduced_data_test = pd.concat([reduced_data_test, OH_reduced_test], axis=1)

In [151]:
reduced_data_train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,FamilySize,...,Cabin_Letter_Unknown,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,Embarked_C,Embarked_Q,Embarked_S,Embarked_nan
0,1,0,3,"Braund, Mr. Owen Harris",1,22.0,1,0,7.2500,2,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",0,38.0,1,0,71.2833,2,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
2,3,1,3,"Heikkinen, Miss. Laina",0,26.0,0,0,7.9250,1,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",0,35.0,1,0,53.1000,2,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,5,0,3,"Allen, Mr. William Henry",1,35.0,0,0,8.0500,1,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [152]:
reduced_data_test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Fare,FamilySize,Cabin_Letter_A,...,Cabin_Letter_Unknown,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,Embarked_C,Embarked_Q,Embarked_S,Embarked_nan
0,892,3,"Kelly, Mr. James",1,34.5,0,0,7.8292,1,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",0,47.0,1,0,7.0000,2,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,894,2,"Myles, Mr. Thomas Francis",1,62.0,0,0,9.6875,1,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3,895,3,"Wirz, Mr. Albert",1,27.0,0,0,8.6625,1,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",0,22.0,1,1,12.2875,3,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [153]:
# Number of missing values in each column of training data
missing_val_count_by_column_train = (reduced_data_train.isnull().sum())
print(missing_val_count_by_column_train[missing_val_count_by_column_train > 0])

Series([], dtype: int64)


In [154]:
# Number of missing values in each column of test data
missing_val_count_by_column_test = (reduced_data_test.isnull().sum())
print(missing_val_count_by_column_test[missing_val_count_by_column_test > 0])

Series([], dtype: int64)


Quiero ver el nombre de las columnas para luego saber cuáles usar para el RF y el XGB.

In [155]:
reduced_data_train.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Fare', 'FamilySize', 'Cabin_Letter_A', 'Cabin_Letter_B',
       'Cabin_Letter_C', 'Cabin_Letter_D', 'Cabin_Letter_E', 'Cabin_Letter_F',
       'Cabin_Letter_G', 'Cabin_Letter_T', 'Cabin_Letter_Unknown',
       'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare',
       'Embarked_C', 'Embarked_Q', 'Embarked_S', 'Embarked_nan'],
      dtype='object')

In [156]:
reduced_data_test.columns

Index(['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'FamilySize', 'Cabin_Letter_A', 'Cabin_Letter_B', 'Cabin_Letter_C',
       'Cabin_Letter_D', 'Cabin_Letter_E', 'Cabin_Letter_F', 'Cabin_Letter_G',
       'Cabin_Letter_T', 'Cabin_Letter_Unknown', 'Title_Master', 'Title_Miss',
       'Title_Mr', 'Title_Mrs', 'Title_Rare', 'Embarked_C', 'Embarked_Q',
       'Embarked_S', 'Embarked_nan'],
      dtype='object')

## Random Forests

In [157]:
from sklearn.tree import DecisionTreeClassifier # Classifier para discreto, Regressor para continuo

# Guardamos el predicition target 
y_train = reduced_data_train.Survived 
X_train = reduced_data_train[['Sex','Age','Fare','Pclass','FamilySize','Cabin_Letter_A','Cabin_Letter_B',
                              'Cabin_Letter_C','Cabin_Letter_D','Cabin_Letter_E','Cabin_Letter_F','Cabin_Letter_G',
                              'Cabin_Letter_T','Cabin_Letter_Unknown','Title_Master','Title_Miss','Title_Mr','Title_Mrs',
                              'Title_Rare','Embarked_C','Embarked_Q','Embarked_S','Embarked_nan']] 

X_test = reduced_data_test[['Sex','Age','Fare','Pclass','FamilySize','Cabin_Letter_A','Cabin_Letter_B',
                              'Cabin_Letter_C','Cabin_Letter_D','Cabin_Letter_E','Cabin_Letter_F','Cabin_Letter_G',
                              'Cabin_Letter_T','Cabin_Letter_Unknown','Title_Master','Title_Miss','Title_Mr','Title_Mrs',
                              'Title_Rare','Embarked_C','Embarked_Q','Embarked_S','Embarked_nan']]  

# Define model. Specify a number for random_state to ensure same results each run
titanic_model = DecisionTreeClassifier(max_leaf_nodes=50,random_state=1,criterion='gini',max_depth=7)

# Fit model
titanic_model.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=7, max_leaf_nodes=50, random_state=1)

In [158]:
predictions = titanic_model.predict(X_test)

## Saving the solutions for Random Forest

In [159]:
solution = pd.DataFrame({'PassengerId': reduced_data_test.PassengerId, 'Survived': predictions})
solution.to_csv('submission.csv', index=False)

colsurv_rf = solution['Survived']

## XGBoost

Necesitamos primero definir X_train y y_train y luego dividirlo con split para poder hacer la validación con XGBoost.

In [160]:
from sklearn.model_selection import train_test_split

# Dividimos el train original en train + valid
X_train_new, X_valid, y_train_new, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
#test_size = 0.2 implica que el 20% de X_train va para X_valid (test)

Generamos el modelo:

In [161]:
from xgboost import XGBClassifier

titanic_xgmodel = XGBClassifier(
    n_estimators=1000,        # número de árboles
    learning_rate=0.05,      # tamaño de los pasos de boosting
    max_depth=4,             # profundidad máxima de cada árbol
    subsample=0.8,           # fracción de datos por árbol
    colsample_bytree=0.8,    # fracción de features por árbol
    use_label_encoder=False,
    eval_metric="logloss",
    early_stopping_rounds=20, 
    verbose=False,
    objective="binary:logistic",
    min_child_weight= 7)

titanic_xgmodel.fit(X_train_new, y_train_new, eval_set=[(X_valid, y_valid)])

[0]	validation_0-logloss:0.66050
[1]	validation_0-logloss:0.64176
[2]	validation_0-logloss:0.62408


/opt/anaconda3/lib/python3.13/site-packages/xgboost/callback.py:386: UserWarning: [17:23:15] WARNING: /var/folders/sy/f16zz6x50xz3113nwtb9bvq00000gp/T/abs_7eg2b3w9sf/croot/xgboost-split_1749630922962/work/src/learner.cc:738: 
Parameters: { "use_label_encoder", "verbose" } are not used.

  self.starting_round = model.num_boosted_rounds()


[3]	validation_0-logloss:0.60979
[4]	validation_0-logloss:0.59471
[5]	validation_0-logloss:0.58220
[6]	validation_0-logloss:0.57002
[7]	validation_0-logloss:0.55905
[8]	validation_0-logloss:0.54927
[9]	validation_0-logloss:0.53992
[10]	validation_0-logloss:0.53038
[11]	validation_0-logloss:0.52221
[12]	validation_0-logloss:0.51567
[13]	validation_0-logloss:0.50815
[14]	validation_0-logloss:0.50238
[15]	validation_0-logloss:0.49724
[16]	validation_0-logloss:0.49138
[17]	validation_0-logloss:0.48573
[18]	validation_0-logloss:0.48024
[19]	validation_0-logloss:0.47570
[20]	validation_0-logloss:0.47284
[21]	validation_0-logloss:0.46875
[22]	validation_0-logloss:0.46563
[23]	validation_0-logloss:0.46227
[24]	validation_0-logloss:0.45860
[25]	validation_0-logloss:0.45535
[26]	validation_0-logloss:0.45251
[27]	validation_0-logloss:0.45041
[28]	validation_0-logloss:0.44761
[29]	validation_0-logloss:0.44479
[30]	validation_0-logloss:0.44318
[31]	validation_0-logloss:0.44144
[32]	validation_0-log

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=20,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=7, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1000, n_jobs=None,
              num_parallel_tree=None, ...)

In [162]:
#from sklearn.model_selection import GridSearchCV

# Definir el modelo base
#xgb = XGBClassifier(
#    random_state=42,
#    use_label_encoder=False,
#    eval_metric='logloss'   # evita warnings
#)

# Definir el grid de hiperparámetros
#param_grid = {
#    'n_estimators': [1000],   # número de árboles
#    'max_depth': [3, 4, 5,],        # profundidad de los árboles
#    'learning_rate': [0.01, 0.05 ,0.1], # tasa de aprendizaje
#    'subsample': [0.8, 1.0],      # porcentaje de datos usados en cada árbol
#    'colsample_bytree': [0.8, 1.0], # porcentaje de features usados en cada árbol
#    'min_child_weight': [6,7,8],

#}

# Configurar GridSearchCV
#grid_search = GridSearchCV(
#    estimator=xgb,
#    param_grid=param_grid,
#    scoring='roc_auc',  # 'accuracy' # puedes cambiar a 'roc_auc' si tu problema es binario
#    cv=5,
#    verbose=2,
#    n_jobs=-1
#)

# Entrenar
#grid_search.fit(X_train_new, y_train_new)

# Mostrar mejores resultados
#print("Mejores parámetros:", grid_search.best_params_)
#print("Mejor score (CV):", grid_search.best_score_)

In [163]:
predictions_xgb = titanic_xgmodel.predict(X_test)

## Saving the solutions for XGBoost

In [164]:
solution = pd.DataFrame({'PassengerId': reduced_data_test.PassengerId, 'Survived': predictions_xgb})
solution.to_csv('submission.csv', index=False)

colsurv_xgb = solution['Survived']

## Comparing Random Forest and XGBoost solutions with the exact ones

In [165]:
perfect_solution = pd.read_csv("/Users/alvaromartinezmartinez/Desktop/Python/MLJupyter/checksubmission_titanic.csv")

colsurv_perf = perfect_solution['Survived']

In [166]:
diff_rf = colsurv_rf != colsurv_perf
diff_xgb = colsurv_xgb != colsurv_perf  # True where values differ

total_cells_rf = diff_rf.size              # total number of cells (418)
total_cells_xgb = diff_xgb.size   

num_differences_rf = diff_rf.sum().sum()
num_differences_xgb = diff_xgb.sum().sum()

percent_equal_rf = 100-((num_differences_rf / total_cells_rf) * 100)
percent_equal_xgb = 100-((num_differences_xgb / total_cells_xgb) * 100)

print(f"Percentage of equal cells for Random Forest: {percent_equal_rf:.2f}%")
print(f"Percentage of equal cells for XGBoost: {percent_equal_xgb:.2f}%")

Percentage of equal cells for Random Forest: 76.08%
Percentage of equal cells for XGBoost: 78.47%
